# Launch: End-to-End Claims Processing Demo

This notebook provisions the demo infrastructure and starts the Streamlit app.

1. Run all cells to set up resources and launch the app
2. Interact with the demo in the Streamlit UI
3. Run the **Cleanup** cell when done to tear down demo resources

## Step 0 — Install Dependencies

In [ ]:
!pip install -q -r requirements.txt

**Restart the kernel after installing**, then continue from Step 1.

## Step 1 — Provision Demo Infrastructure

This creates a DynamoDB table, Step Functions state machine, IAM role, and AgentCore Memory resources — all `demo-` prefixed.

In [ ]:
import boto3
import json
import time

REGION = 'us-east-1'
DYNAMODB_TABLE = 'demo-claims-metadata'
SFN_ROLE_NAME = 'demo-claims-hitl-sfn-role'
STATE_MACHINE_NAME = 'demo-claims-hitl-adjudication'
ACTIVITY_NAME = 'demo-claims-human-review'
MEMORY_NAME = 'Demo_IntakeOrchestrator_Memory'

dynamodb = boto3.resource('dynamodb', region_name=REGION)
iam_client = boto3.client('iam', region_name=REGION)
sfn_client = boto3.client('stepfunctions', region_name=REGION)
sts_client = boto3.client('sts', region_name=REGION)
ACCOUNT_ID = sts_client.get_caller_identity()['Account']

# ── 1. DynamoDB table ────────────────────────────────────────────────────────
try:
    table = dynamodb.create_table(
        TableName=DYNAMODB_TABLE,
        KeySchema=[{"AttributeName": "claim_id", "KeyType": "HASH"}],
        AttributeDefinitions=[{"AttributeName": "claim_id", "AttributeType": "S"}],
        BillingMode="PAY_PER_REQUEST",
    )
    table.wait_until_exists()
    print(f"✅ Created DynamoDB table: {DYNAMODB_TABLE}")
except Exception as e:
    if "ResourceInUseException" in str(e):
        print(f"ℹ️  DynamoDB table already exists: {DYNAMODB_TABLE}")
    else: raise

# ── 2. IAM role for Step Functions ────────────────────────────────────────────
trust_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "states.amazonaws.com"}, "Action": "sts:AssumeRole"}]
})
try:
    role_resp = iam_client.create_role(RoleName=SFN_ROLE_NAME, AssumeRolePolicyDocument=trust_policy)
    SFN_ROLE_ARN = role_resp["Role"]["Arn"]
    print(f"✅ Created IAM role: {SFN_ROLE_NAME}")
    time.sleep(10)
except iam_client.exceptions.EntityAlreadyExistsException:
    SFN_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{SFN_ROLE_NAME}"
    print(f"ℹ️  IAM role already exists: {SFN_ROLE_NAME}")

# ── 3. Step Functions Activity + State Machine ───────────────────────────────
# Create an Activity (the human review task)
try:
    activity_resp = sfn_client.create_activity(name=ACTIVITY_NAME)
    ACTIVITY_ARN = activity_resp["activityArn"]
    print(f"✅ Created activity: {ACTIVITY_NAME}")
except Exception as e:
    ACTIVITY_ARN = f"arn:aws:states:{REGION}:{ACCOUNT_ID}:activity:{ACTIVITY_NAME}"
    print(f"ℹ️  Activity may already exist: {ACTIVITY_NAME}")

# State machine with Activity-based wait
sm_def = json.dumps({
    "Comment": "Demo: Claims HITL - waits for human via Activity",
    "StartAt": "WaitForHumanReview",
    "States": {
        "WaitForHumanReview": {
            "Type": "Task",
            "Resource": ACTIVITY_ARN,
            "TimeoutSeconds": 86400,
            "ResultPath": "$.human_decision",
            "End": True
        }
    }
})

try:
    sm_resp = sfn_client.create_state_machine(
        name=STATE_MACHINE_NAME,
        definition=sm_def,
        roleArn=SFN_ROLE_ARN,
        type='STANDARD',
    )
    STATE_MACHINE_ARN = sm_resp["stateMachineArn"]
    print(f"✅ Created state machine: {STATE_MACHINE_NAME}")
except sfn_client.exceptions.StateMachineAlreadyExists:
    STATE_MACHINE_ARN = f"arn:aws:states:{REGION}:{ACCOUNT_ID}:stateMachine:{STATE_MACHINE_NAME}"
    print(f"ℹ️  State machine already exists: {STATE_MACHINE_NAME}")

# Save ARNs for the app
import os
with open('.demo_config.json', 'w') as f:
    json.dump({
        "STATE_MACHINE_ARN": STATE_MACHINE_ARN,
        "ACTIVITY_ARN": ACTIVITY_ARN,
    }, f)

# ── 4. AgentCore Memory (STM + LTM) ─────────────────────────────────────────
from bedrock_agentcore.memory import MemoryClient
memory_client = MemoryClient(region_name=REGION)

try:
    ltm_memory = memory_client.create_memory_and_wait(
        name=MEMORY_NAME,
        description="Demo: Intake Orchestrator memory with STM and LTM",
        strategies=[
            {
                "summaryMemoryStrategy": {
                    "name": "ClaimSessionSummarizer",
                    "namespaceTemplates": ["/summaries/{actorId}/{sessionId}/"]
                }
            },
            {
                "semanticMemoryStrategy": {
                    "name": "ClaimFactExtractor",
                    "namespaceTemplates": ["/facts/{actorId}/"]
                }
            },
        ]
    )
    DEMO_MEMORY_ID = ltm_memory.get("id")
    print(f"✅ Created AgentCore Memory: {MEMORY_NAME} (ID: {DEMO_MEMORY_ID})")
except Exception as e:
    if "already exists" in str(e):
        print(f"ℹ️  Memory already exists, looking up...")
        DEMO_MEMORY_ID = ''
        for m in memory_client.list_memories():
            if m.get("name") == MEMORY_NAME:
                DEMO_MEMORY_ID = m.get("id")
                break
        print(f"   Memory ID: {DEMO_MEMORY_ID}")
    else:
        raise

# Save memory ID for the Streamlit app
with open('.demo_memory_id', 'w') as f:
    f.write(DEMO_MEMORY_ID)

print()
print("✅ All demo infrastructure ready")

## Step 2 — Start the Streamlit App

Run the cell below to start the demo app. A URL will be printed — open it in a new browser tab.

The AgentCore Memory is provisioning in the background (takes 1-2 minutes for LTM strategies). The app will work immediately for claim submission; the Memory tab will populate once LTM is active.

In [ ]:
import subprocess
import os

print('Starting Streamlit app...')
print()
print('If running in SageMaker, the app will be available at:')
print('  https://<your-domain>/proxy/8501/')
print()
print('Press the stop button (interrupt kernel) to stop the app.')

In [ ]:
!streamlit run app.py --server.port 8501 --server.headless true

## Cleanup (Optional)

Run this cell to tear down **all** demo resources. This does NOT affect notebooks 01-05.

In [ ]:
import boto3
import shutil
from pathlib import Path

REGION = 'us-east-1'
DYNAMODB_TABLE = 'demo-claims-metadata'
SFN_ROLE_NAME = 'demo-claims-hitl-sfn-role'
STATE_MACHINE_NAME = 'demo-claims-hitl-adjudication'
MEMORY_NAME = 'Demo_IntakeOrchestrator_Memory'

sts_client = boto3.client('sts', region_name=REGION)
ACCOUNT_ID = sts_client.get_caller_identity()['Account']
STATE_MACHINE_ARN = f'arn:aws:states:{REGION}:{ACCOUNT_ID}:stateMachine:{STATE_MACHINE_NAME}'

sfn_client = boto3.client('stepfunctions', region_name=REGION)
iam_client = boto3.client('iam', region_name=REGION)
dynamodb_client = boto3.client('dynamodb', region_name=REGION)

# Delete state machine
try:
    sfn_client.delete_state_machine(stateMachineArn=STATE_MACHINE_ARN)
    print(f"✅ Deleted state machine")
except Exception as e:
    print(f"⚠️  {e}")

# Delete IAM role
try:
    iam_client.delete_role_policy(RoleName=SFN_ROLE_NAME, PolicyName='demo-hitl-dynamodb')
    iam_client.delete_role(RoleName=SFN_ROLE_NAME)
    print(f"✅ Deleted IAM role")
except Exception as e:
    print(f"⚠️  {e}")

# Delete DynamoDB table
try:
    dynamodb_client.delete_table(TableName=DYNAMODB_TABLE)
    print(f"✅ Deleted DynamoDB table")
except Exception as e:
    print(f"⚠️  {e}")

# Delete AgentCore Memory
try:
    from bedrock_agentcore.memory import MemoryClient
    mc = MemoryClient(region_name=REGION)
    for m in mc.list_memories():
        if m.get("name") == MEMORY_NAME:
            mc.delete_memory(memory_id=m["id"])
            print(f"✅ Deleted AgentCore Memory")
            break
except Exception as e:
    print(f"⚠️  {e}")

# Remove local folders and temp files
for item in ['demo_claim_documents', 'demo_extracted_output', '.demo_memory_id']:
    p = Path(item)
    if p.is_dir():
        shutil.rmtree(p)
        print(f'✅ Removed {item}/')
    elif p.is_file():
        p.unlink()
        print(f'✅ Removed {item}')

print()
print("✅ Demo cleanup complete")

In [ ]:
]